In [0]:
cus_df = spark.table("samples.tpch.customer")
cus_df.display()

In [0]:
 ord§ers_df= spark.table("samples.tpch.orders")
 orders_df.display()

In [0]:
# -- custkey, name, mobile num, how many orders cust has placed
# -- Join orders_df & cus_df to get Order Information

joined_df = (
    cus_df.join(orders_df, on =cus_df.c_custkey == orders_df.o_custkey,
                how = "left")
)

joined_df.display()




In [0]:
# Optimized way
# -- custkey, name, mobile num, how many orders cust has placed
# -- Join orders_df & cus_df to get Order Information

joined_df = (
    cus_df.select("c_custkey","c_name","c_phone")
    .join(
        orders_df.select("o_orderkey","o_custkey"),
           on = cus_df.c_custkey == orders_df.o_custkey,
                how = "left"
        )
        .drop('o_custkey') #Dropping unwanted column after Join, in this case o_custkey
)

joined_df.display()




In [0]:
from pyspark.sql import functions as F
result_df = (
    joined_df.groupBy("c_custkey")
    .agg(
        F.count("o_orderkey").alias("total_orders"),
        F.first("c_name").alias("name"),
        F.first("c_phone").alias("phone")
    )
    .orderBy(F.desc("total_orders"))
)

result_df.display()


In [0]:
# Deriving VIP customers
# any customers who placed more than 20 Orders Should be picked as VIP customer
# now De Derive a new column called is_vip_cus, if a customer has placed total_orders > 20 then True else False

In [0]:
result_df = (
    result_df.withColumn("is_vip_cus", 
                         F.when(F.col("total_orders")>20,True)
                         .otherwise(False))
)

result_df.display()

In [0]:
result_df = (
    result_df.withColumn("cus_Type", 
                         F.when(F.col("total_orders")>40,"Super_Valued")
                         .when(F.col("total_orders")>20,"Valued")
                         .otherwise("Regular"))
)

result_df.display()